# Importador histórico ChileCompra — versión multitabla experimental

[Volver al README](https://github.com/juliopez/MercadoPublico-Inteligencia#readme)

**Destino exclusivamente:** clon de Google Sheets. **No invoca API ChileCompra ni GitHub Actions.**

Esta versión admite una tabla de destino por ejecución: `Procesos`, `Ofertas`, `Resultados` y `OrdenesCompra`. Las hojas `Control`, `ControlV2`, `Indicadores` y el resumen histórico no se cargan: no son tablas de importación independientes.

**Seguridad:** inicia en simulación, exige columnas y claves probatorias mínimas, no inventa proveedores/adjudicaciones, no actualiza registros existentes, genera reporte de calidad y requiere confirmación explícita para escribir. Esta ampliación aún necesita contrastar columnas de archivos reales de ofertas, adjudicaciones y OC.

In [ ]:
!pip -q install pandas gspread google-api-python-client google-auth
from google.colab import auth, files
from google.auth import default
from googleapiclient.discovery import build
import pandas as pd, gspread, io, zipfile, csv, re, unicodedata, os, json, time, datetime
auth.authenticate_user()
creds,_=default(scopes=['https://www.googleapis.com/auth/spreadsheets','https://www.googleapis.com/auth/drive'])
gc=gspread.authorize(creds)
drive=build('drive','v3',credentials=creds)
print('Autenticación lista')

## Configuración — una tabla por archivo y ejecución
No cambies `CLON_ID`. `ESCRIBIR=False` significa **sin modificaciones**.

In [ ]:
CLON_ID='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
TABLA='Ofertas'   # Procesos | Ofertas | Resultados | OrdenesCompra
FUENTE='compra_agil'  # licitaciones | compra_agil | ordenes_compra
ESCRIBIR=False
CONFIRMACION=''
LOTE=250
assert CLON_ID=='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
assert TABLA in ('Procesos','Ofertas','Resultados','OrdenesCompra')
assert FUENTE in ('licitaciones','compra_agil','ordenes_compra')
assert (TABLA,FUENTE) not in [('Ofertas','ordenes_compra'),('OrdenesCompra','licitaciones'),('OrdenesCompra','compra_agil')]
libro=gc.open_by_key(CLON_ID)
ws=libro.worksheet(TABLA)

In [ ]:
ESQUEMAS={
'Procesos':['codigo','nombre','estado','convocatoria','fecha_publicacion','fecha_cierre','monto_disponible_clp','organismo_comprador','region','total_ofertas_recibidas','motivo_seleccion','url_detalle','fecha_sincronizacion','categoria','subcategoria','modalidad'],
'Ofertas':['codigo','proveedor','monto_ofertado_clp','fecha_oferta','fuente','id_cotizacion','seleccionado'],
'Resultados':['codigo','adjudicatario','monto_adjudicado_clp','fecha_resultado','motivo_seleccion','fuente','validado','id_cotizacion_ganadora','monto_oferta_seleccionada_clp','estado_verificacion_oc'],
'OrdenesCompra':['codigo_oc','codigo_proceso','fecha_creacion','fecha_aceptacion','estado','proveedor','monto_total_oc_clp','tipo_oc','cantidad_lineas','total_lineas_neto']}
assert ws.row_values(1)==ESQUEMAS[TABLA], 'Cabeceras incompatibles: abortado'
def norm(v):
 s=unicodedata.normalize('NFKD',str(v))
 return re.sub('[^a-z0-9]','', ''.join(c for c in s if not unicodedata.combining(c)).lower())
def val(x):
 s=str(x).strip()
 return '' if s.lower() in ('nan','null','none','nat','<na>') else s
def cod(x):return val(x).upper()
def fecha(x):
 s=val(x)
 if not s:return ''
 try:
  d=pd.to_datetime(s,errors='coerce',dayfirst=not bool(re.match(r'^\d{4}-',s)))
  return d.strftime('%Y-%m-%d') if pd.notna(d) else ''
 except Exception:return ''
def monto(x):
 s=val(x).replace('$','').replace(' ','')
 if not s:return ''
 if re.fullmatch(r'-?\d{1,3}(\.\d{3})+(,\d+)?',s):s=s.replace('.','').replace(',','.')
 elif ',' in s and '.' not in s:s=s.replace(',','.')
 try:return str(float(s)) if '.' in s else str(int(s))
 except ValueError:return ''
ALIAS={
'codigo':['CodigoExterno','CodigoCotizacion','CodigoProceso','CodigoLicitacion','Codigo'],
'nombre':['Nombre','NombreLicitacion','NombreCotizacion','Titulo'],
'estado':['Estado','EstadoLicitacion','EstadoOC'],
'fecha_publicacion':['FechaPublicacion','FechaCreacion'],
'fecha_cierre':['FechaCierre','FechaFinalizacion'],
'monto_disponible_clp':['MontoEstimado','MontoDisponible','Presupuesto'],
'organismo_comprador':['NombreOrganismo','OrganismoComprador'],
'url_detalle':['Link','UrlDetalle'],
'proveedor':['NombreProveedor','Proveedor','RazonSocial','NombreRazonSocial'],
'monto_ofertado_clp':['MontoOfertado','MontoOferta','MontoCotizacion','Monto'],
'fecha_oferta':['FechaOferta','FechaCotizacion'],
'id_cotizacion':['IdCotizacion','CodigoCotizacionDetalle','IdOferta','CodigoOferta'],
'seleccionado':['Seleccionado','Adjudicado','EsGanador'],
'adjudicatario':['Adjudicatario','NombreAdjudicatario','ProveedorAdjudicado'],
'monto_adjudicado_clp':['MontoAdjudicado','MontoAdjudicacion'],
'fecha_resultado':['FechaAdjudicacion','FechaResultado'],
'motivo_seleccion':['MotivoSeleccion','JustificacionAdjudicacion'],
'id_cotizacion_ganadora':['IdCotizacionGanadora','CodigoCotizacionGanadora'],
'codigo_oc':['CodigoOC','CodigoOrdenCompra','CodigoOrdenDeCompra','Codigo'],
'codigo_proceso':['CodigoLicitacion','CodigoProceso','CodigoExternoProceso'],
'fecha_creacion':['FechaCreacion','FechaEnvio','FechaOC'],
'fecha_aceptacion':['FechaAceptacion','fechaAceptacion'],
'monto_total_oc_clp':['MontoTotalOC_PesosChilenos','MontoTotalOC','MontoTotal','TotalOC'],
'tipo_oc':['TipoOC','TipoOrdenCompra'],
'cantidad_lineas':['CantidadLineas','CantidadItems'],
'total_lineas_neto':['TotalLineasNeto','MontoNeto'],
}
# Claves obligatorias: oferta debe tener id individual; resultado necesita evidencia explícita.
OBLIGATORIOS={
'Procesos':['codigo','nombre'],
'Ofertas':['codigo','proveedor','monto_ofertado_clp','id_cotizacion'],
'Resultados':['codigo','adjudicatario','monto_adjudicado_clp'],
'OrdenesCompra':['codigo_oc','proveedor','monto_total_oc_clp']}
CLAVES={'Procesos':['codigo'],'Ofertas':['id_cotizacion'],'Resultados':['codigo'],'OrdenesCompra':['codigo_oc']}
print('Esquema verificado:',TABLA,ESQUEMAS[TABLA])

## Lectura del ZIP y control de correspondencias

Se detiene si el histórico no incluye los campos probatorios de la tabla. No se usarán datos de una cotización para inventar adjudicaciones. Para Ofertas, debe haber ID de cotización único. Los formatos nuevos requieren revisar el reporte antes de habilitar escritura.

In [ ]:
subidos=files.upload()
assert len(subidos)==1,'Sube un solo ZIP o CSV'
nombre=next(iter(subidos))
assert nombre.lower().endswith(('.zip','.csv'))
def contenidos(nombre,bytes_):
 if nombre.lower().endswith('.csv'):yield nombre,bytes_;return
 with zipfile.ZipFile(io.BytesIO(bytes_)) as z:
  csvs=[x for x in z.namelist() if x.lower().endswith('.csv') and not x.startswith('__MACOSX/')]
  assert csvs,'No hay CSV en el ZIP'
  for n in csvs:yield n,z.read(n)
def detector(raw):
 # Validar TODO el CSV: la muestra inicial puede ser cp1252 y fallar al avanzar.
 # latin-1 es fallback sin errores de decodificación, pero se registra el encoding.
 for enc in ('utf-8-sig','utf-8','cp1252','latin-1'):
  try:
   texto=raw.decode(enc,errors='strict')
  except UnicodeDecodeError:
   continue
  linea=texto.splitlines()[0] if texto.splitlines() else ''
  sep=max([';',',','\t','|'],key=linea.count)
  if linea.count(sep)>0:return enc,sep
 raise ValueError('CSV sin encabezado o delimitador reconocible')
def equivalencias(cols):
 d={norm(x):x for x in cols}
 return {k:next((d[norm(a)] for a in aliases if norm(a) in d),None) for k,aliases in ALIAS.items()}
def transformar(row,m):
 r={k:val(row.get(m.get(k),'') if m.get(k) else '') for k in ESQUEMAS[TABLA]}
 for k in ['codigo','codigo_oc','codigo_proceso']: 
  if k in r:r[k]=cod(r[k])
 for k in r:
  if k.startswith('fecha_'):r[k]=fecha(r[k])
  if k.startswith('monto_') or k=='total_lineas_neto':r[k]=monto(r[k])
 if TABLA=='Procesos':r['modalidad']='Compra Ágil' if FUENTE=='compra_agil' else 'Licitación'
 if TABLA in ('Ofertas','Resultados'):r['fuente']='Histórico ChileCompra '+FUENTE+' '+nombre
 if TABLA=='Resultados':
  r['validado']='Pendiente de verificación de OC'
  r['estado_verificacion_oc']='OC no verificada'
 return r
filas=0;invalidas=0;detalle={};diagnosticos=[]
for item,raw in contenidos(nombre,subidos[nombre]):
 enc,sep=detector(raw)
 first=True;cont=0
 for chunk in pd.read_csv(io.BytesIO(raw),encoding=enc,sep=sep,dtype=str,keep_default_na=False,chunksize=20000,on_bad_lines='error'):
  if first:
   encabezados={norm(c) for c in chunk.columns}
   if 'montototalocpesoschilenos' in encabezados and TABLA!='OrdenesCompra':
    raise ValueError('Este CSV corresponde a Órdenes de Compra. Configura TABLA="OrdenesCompra" y FUENTE="ordenes_compra".')
   if TABLA=='OrdenesCompra' and 'montototalocpesoschilenos' not in encabezados and not ({'montototaloc','montototal','totaloc'} & encabezados):
    raise ValueError('No se reconoce un monto total de OC. Comprueba que el ZIP sea de Órdenes de Compra.')
   m=equivalencias(chunk.columns)
   missing=[k for k in OBLIGATORIOS[TABLA] if not m.get(k)]
   assert not missing, f'{item}: faltan columnas esenciales {missing}. Encontradas: {list(chunk.columns)}'
   diagnosticos.append({'archivo':item,'codificacion':enc,'delimitador':sep,'columnas':m})
   first=False
  for row in chunk.to_dict('records'):
   filas+=1;cont+=1
   r=transformar(row,m)
   if any(not r.get(k) for k in OBLIGATORIOS[TABLA]):invalidas+=1;continue
   clave=tuple(r[k] for k in CLAVES[TABLA])
   prev=detalle.get(clave)
   if prev is not None:
    # Ofertas duplicadas con mismo ID y valores diferentes => detener.
    if TABLA=='Ofertas' and (prev['codigo']!=r['codigo'] or prev['proveedor']!=r['proveedor'] or prev['monto_ofertado_clp']!=r['monto_ofertado_clp']):
     raise ValueError('Conflicto de datos para oferta '+repr(clave))
    continue
   detalle[clave]=r
 print(item,cont,'filas')
assert detalle,'Sin registros válidos'
print('Preparación completada')

In [ ]:
# Se leen claves de la hoja real, aun si el clon está vacío.
col_indices=[ESQUEMAS[TABLA].index(c)+1 for c in CLAVES[TABLA]]
cols_existentes=[ws.col_values(i)[1:] for i in col_indices]
n=max(map(len,cols_existentes),default=0)
existentes=[]
for i in range(n):
 key=tuple(cod(col[i]) if i<len(col) else '' for col in cols_existentes)
 if all(key):existentes.append(key)
assert len(existentes)==len(set(existentes)),'Hay claves duplicadas en el destino'
previas=set(existentes)
nuevos=[r for k,r in detalle.items() if k not in previas]
reporte={'archivo':nombre,'fuente':FUENTE,'tabla':TABLA,'filas_originales':filas,'filas_invalidas':invalidas,
         'claves_unicas_validas':len(detalle),'duplicados_en_origen':filas-invalidas-len(detalle),
         'ya_existentes':len(detalle)-len(nuevos),'nuevos':len(nuevos),
         'total_destino_inicial':len(previas),'total_destino_esperado':len(previas)+len(nuevos),
         'mapeos':diagnosticos}
print(json.dumps(reporte,indent=2,ensure_ascii=False))
display(pd.DataFrame(nuevos[:5],columns=ESQUEMAS[TABLA]))
stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
with open('/content/reporte_'+stamp+'.json','w') as f:json.dump(reporte,f,ensure_ascii=False,indent=2)
pd.DataFrame(nuevos,columns=ESQUEMAS[TABLA]).to_csv('/content/nuevos_'+stamp+'.csv',index=False,encoding='utf-8-sig')

## Escritura opcional en el clon
Para esta primera versión multitabla, realizar primero simulación de un histórico real y validar columnas. Si corresponde, activar escritura de una tabla con confirmación y respaldo. Nunca ejecutar al mismo tiempo dos instancias.

In [ ]:
if not ESCRIBIR:
 print('SIMULACIÓN: Google Sheets no fue modificado.')
else:
 assert CONFIRMACION=='AUTORIZO CARGA EN CLON'
 assert CLON_ID=='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
 assert ws.row_values(1)==ESQUEMAS[TABLA]
 copia=drive.files().copy(fileId=CLON_ID,body={'name':'Respaldo multitabla clon '+stamp},fields='id,webViewLink').execute()
 assert copia['id']!=CLON_ID
 print('RESPALDO:',copia.get('webViewLink',copia['id']))
 escritos=0
 for i in range(0,len(nuevos),LOTE):
  # Releer claves para reanudación idempotente (sin ejecución concurrente).
  columnas=[ws.col_values(j)[1:] for j in col_indices]
  count=max(map(len,columnas),default=0)
  current=set(tuple(cod(c[x]) if x<len(c) else '' for c in columnas) for x in range(count))
  assert len(current)==count,'Clave ausente o duplicada detectada'
  batch=[r for r in nuevos[i:i+LOTE] if tuple(r[k] for k in CLAVES[TABLA]) not in current]
  if not batch:continue
  pos=len(ws.col_values(1))+1
  ws.update(range_name=f'A{pos}:{chr(64+len(ESQUEMAS[TABLA]))}{pos+len(batch)-1}',values=[[str(r.get(c,'')) for c in ESQUEMAS[TABLA]] for r in batch],value_input_option='RAW')
  # Verificar todas las claves recién insertadas.
  nuevo_lectura=ws.get(f'A{pos}:{chr(64+len(ESQUEMAS[TABLA]))}{pos+len(batch)-1}')
  claves={tuple(cod(f[ESQUEMAS[TABLA].index(k)]) for k in CLAVES[TABLA]) for f in nuevo_lectura}
  assert claves=={tuple(r[k] for k in CLAVES[TABLA]) for r in batch}
  escritos+=len(batch)
  print('Escritos:',escritos)
  time.sleep(1)
 print('Terminada la carga de',TABLA,':',escritos,'nuevos')